# Trimer Case2:  Shared codes

This page contains parameter values and functions used in other pages of the chapter.

In [31]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import *

**Constant parameters**

In [32]:
# default energy scale
omega0 = 1.0

# default spontaneous emission rate
gamma0 = 1.e-8

def scale(omega0):
    k0 = omega0  # omega = c k, c=1
    lambda0 = 2*np.pi/k0
    return k0, lambda0

def planck_dist(omega,T):
    return 1/(np.exp(omega/T)-1) # Planck distributio

**Spin operators for 2 emitters**

In [33]:
# Spin operators ---
def spin_ops():
    i2 = qeye(2)
    sz = [tensor(sigmaz(),i2),tensor(i2,sigmaz())]
    sp = [tensor(sigmap(),i2),tensor(i2,sigmap())]
    sm = [tensor(sigmam(),i2),tensor(i2,sigmam())]
    return sz, sp, sm

**Product basis**

In [34]:
def pbasis():
    #--This function calls kprod3--
    # v = product basis vectors [8][8]
    # label = names of corresponding states [8]
    label=["ee","eg","ge","gg"]
    v=[]
    v.append(tensor(basis(2,0),basis(2,0)))
    v.append(tensor(basis(2,0),basis(2,1)))
    v.append(tensor(basis(2,1),basis(2,0)))
    v.append(tensor(basis(2,1),basis(2,1)))

    return v, label

**Dicke basis**

In [35]:
def dicke(v):
    #--- v = product basis [4][4]
    #--- u = Dicke basis [4][4]
    label=["e","s","a","g"]
    u=[]
    u.append(v[0])
    u.append(1/np.sqrt(2)*(v[1]+v[2]))
    u.append(1/np.sqrt(2)*(v[1]-v[2]))
    u.append(v[3]) 
    return u, label

In [36]:
def emitter_pos(a):
    r1 = [a/2,0,0]
    r2 = [-a/2,0,0]
    return [r1,r2]

**Dipole-dipole coupling**


In [37]:
def ddcoupling(x,theta):
    y=(1-np.cos(theta)**2)*np.cos(x)/x + (1-3*np.sin(theta)**2)*(np.sin(x)/x**2 - np.cos(x)/x**3)
    return -y*3/4

**Hamiltonian**

In [38]:
def hamiltonian(Omega):
    sz, sp, sm = spin_ops()
    H = Omega*(sp[0]*sm[1]+sp[1]*sm[0])
    return H

**Decay rates**

In [39]:
def decay_rate(x,theta):
    Gamma = (3/2)*(1-np.cos(theta)**2)*np.sin(x)/x + (1-3*np.cos(theta)**2)*(np.cos(x)/x**2 - np.sin(x)/x**3)
    gamma = [1+Gamma, 1-Gamma]
    return gamma

**Jump and Collaps operators**

In [40]:
def collapse_ops(gamma,nbar):
    # L = emission and absorption operators [6]
    # c_ops = collapse operators including the decay rates [6]
    sz, sp, sm = spin_ops()
    L1=sm[0]
    L2=sm[1]
    L3=L1.dag()
    L4=L2.dag()
    L = [L1,L2,L3,L4]
    c_ops = []
    c_ops.append(np.sqrt(gamma[0]*(nbar+1))*L1)
    c_ops.append(np.sqrt(gamma[1]*(nbar+1))*L2)
    c_ops.append(np.sqrt(gamma[0]*nbar)*L3)
    c_ops.append(np.sqrt(gamma[1]*nbar)*L4)
    return L, c_ops

**Energy eigenvalues**

In [41]:
def eigen_energies(omega0,Omega,gamma0):
    sz, sp, sm = spin_ops()
    E=[omega0,Omega, -Omega,-omega0]
    H=omega0/2*(sz[0]+sz[1])
    H+=Omega*gamma0*(sp[0]*sm[1]+sp[1]*sm[0])
    W=H.eigenenergies()
    order = np.argsort(E, descending=True)
    return np.array(W), np.array(E), order

**Transition energy**

In [42]:
def transition_energies(E):
    # w = transition energies for each channel [3]
    w0 = np.array([E[0]-E[1], E[1]-E[3]])
    w1 = np.array([E[0]-E[2], E[2]-E[3]])
    w = [w0,w1]
    return w

**position of emitters**

In [43]:
def emitters_pos(a):
    r1 = np.array([a/2, 0, 0])
    r2 = np.array([-a/2,0,0])
    return np.array([r1,r2])

**location of detector**

In [44]:
def detector_pos(theta,phi):
    return np.array([np.sin(theta)*np.cos(phi),np.sin(theta)*np.sin(phi),np.cos(theta)])

**Boltzmann distribution**

In [45]:
def get_Gibbs(E,T):
    Boltzmann = [np.exp(-E[k]/T) for k in range(4)]
    Z = sum(Boltzmann)
    return Boltzmann/Z